In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

In [ ]:
DATA_DIR = Path("processed_data_two_codes")

## Load and Combined Data

In [ ]:
# demographics = pd.read_csv(DATA_DIR / 'demo.csv')
demographics = pd.read_csv(DATA_DIR / 'demo_8_17_26.csv')
labs = pd.read_csv(DATA_DIR / 'lab_raw.csv')

In [ ]:
demographics.head()

In [ ]:
labs.columns

In [ ]:
# person_id	age	ethnicity_hispanic	ethnicity_nonhispanic	ethnicity_other	sx_birth_female	sx_birth_male	sx_birth_other	race_asian	race_black	race_mena	race_nhpi	race_other	race_white

In [ ]:
# target_order = [
#     "MRN", "age",
#     "ethnicity_hispanic", "ethnicity_nonhispanic", "ethnicity_other",
#     "sx_birth_female", "sx_birth_male", "sx_birth_other",
#     "race_asian", "race_black", "race_mena", "race_nhpi", "race_other", "race_white",
# ]

# df2 = demographics.copy()

# # 1) ensure missing columns exist
# for col in ["sx_birth_other", "race_mena"]:
#     if col not in df2.columns:
#         df2[col] = 0

# # 2) fold AIAN into race_other
# if "race_aian" in df2.columns:
#     # if race_other doesn't exist, create it first
#     if "race_other" not in df2.columns:
#         df2["race_other"] = 0

#     # add (works for 0/1; if booleans, cast to int)
#     df2["race_other"] = df2["race_other"].astype(int) + df2["race_aian"].astype(int)

#     # keep it binary (0/1) if you want one-hot style
#     df2["race_other"] = (df2["race_other"] > 0).astype(int)

#     # optionally drop race_aian since it's folded in
#     df2 = df2.drop(columns=["race_aian"])

# # 3) reorder (this will raise if something is still missing)
# missing = [c for c in target_order if c not in df2.columns]
# extra = [c for c in df2.columns if c not in set(target_order)]
# print("Missing vs target:", missing)
# print("Extra columns:", extra)

# df2 = df2[target_order]

In [ ]:
# df2

In [ ]:
# age	ethnicity_hispanic	ethnicity_nonhispanic	ethnicity_other	sx_birth_female	sx_birth_male	sx_birth_other	race_asian	race_black	race_mena	race_nhpi	race_other	race_white

In [ ]:
# age	ethnicity_hispanic	ethnicity_nonhispanic	ethnicity_other	sx_birth_female	sx_birth_male	sx_birth_other	race_asian	race_black	race_mena	race_nhpi	race_other	race_white

In [ ]:
labs.head()

In [ ]:
data = demographics.merge(labs, on = 'MRN', how = 'left')
# data = df2.merge(labs, on = 'MRN', how = 'left')

data.head()

In [ ]:
data.shape

In [ ]:
print(f'There is a total of {len(data)} patients in this cohort.')

## Check Missingness

In [ ]:
# no column has more than 80% missing values, so we can keep all the columns
data.isnull().sum()*100/len(data)

## Imputing Missing Values

In [ ]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import MinMaxScaler

In [ ]:
data = data.set_index('MRN')
data.head()

In [ ]:
scaler = MinMaxScaler()
data_scaled = pd.DataFrame(scaler.fit_transform(data), columns = data.columns).set_index(data.index)
data_scaled.head()

In [ ]:
col_stats = pd.DataFrame({
    "min": scaler.data_min_,
    "max": scaler.data_max_,
    "range": scaler.data_range_
}, index=data.columns)
col_stats.to_csv(DATA_DIR / 'lab_scaling_8_17_26.csv')
col_stats

In [ ]:
col_stats = pd.DataFrame({
    "min": scaler.data_min_,
    "max": scaler.data_max_,
    "range": scaler.data_range_,
    "mean": data.mean()
}, index=data.columns)

col_stats.to_csv(DATA_DIR / 'lab_scaling_with_mean_8_17_26.csv')
col_stats

In [ ]:
DATA_DIR

In [ ]:
import json

scaler_json = r'''{"columns": ["age", "ethnicity_hispanic", "ethnicity_nonhispanic", "ethnicity_other", "sx_birth_female", "sx_birth_male", "sx_birth_other", "race_asian", "race_black", "race_mena", "race_nhpi", "race_other", "race_white", "sbp", "dbp", "heart_rate", "bmi", "a1c", "tsh", "total_chol", "ldl_chol", "hdl_chol", "nonhdl_chol", "triglyceride", "white_blood", "red_blood", "hemoglobin", "hematocrit", "platelets", "sodium", "potassium", "choloride", "co2", "albumin", "alk_phos", "bilirubin", "aspartate_trans", "alaine_trans", "blood_urea_nit", "protein", "calcium", "creatinine", "glucose", "ph_urine"], "data_min": [21.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -33.0, 3.0, 0.0, 7.0, 0.0, 0.0, 0.084, 0.0, 0.0, 20.0, 0.0, 5.2, 0.0, 0.0, 0.7, 0.0, 0.0, 3.0, 0.0, 0.0, 1.18, 0.0, 0.0, 0.0], "data_max": [109.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 248.0, 181.0, 591.0, 3234.2, 10000000.0, 10000000.0, 10000000.0, 10000000.0, 10000000.0, 685.0, 10000000.0, 10000000.0, 10000000.0, 100000000.0, 10000000.0, 10000000.0, 169.0, 10000000.0, 130.0, 50.0, 41000.0, 6000.0, 10000000.0, 10000000.0, 10000000.0, 213.0, 10000000.0, 26.0, 274.5, 10000000.0, 10000000.0], "data_range": [88.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 248.0, 181.0, 591.0, 3234.2, 10000000.0, 10000000.0, 10000000.0, 10000033.0, 9999997.0, 685.0, 9999993.0, 10000000.0, 10000000.0, 99999999.916, 10000000.0, 10000000.0, 149.0, 10000000.0, 124.8, 50.0, 41000.0, 5999.3, 10000000.0, 10000000.0, 9999997.0, 213.0, 10000000.0, 24.82, 274.5, 10000000.0, 10000000.0], "scale": [0.011363636363636364, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.004032258064516129, 0.0055248618784530384, 0.001692047377326565, 0.00030919547337826975, 1e-07, 1e-07, 1e-07, 9.9999670001089e-08, 1.00000030000009e-07, 0.00145985401459854, 1.00000070000049e-07, 1e-07, 1e-07, 1.0000000008400001e-08, 1e-07, 1e-07, 0.006711409395973154, 1e-07, 0.008012820512820514, 0.02, 2.4390243902439026e-05, 0.00016668611337989432, 1e-07, 1e-07, 1.00000030000009e-07, 0.004694835680751174, 1e-07, 0.040290088638195005, 0.0036429872495446266, 1e-07, 1e-07], "min": [-0.23863636363636365, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3.299989110035937e-06, -3.00000090000027e-07, 0.0, -7.00000490000343e-07, 0.0, 0.0, -8.400000007056002e-10, 0.0, 0.0, -0.1342281879194631, 0.0, -0.04166666666666667, 0.0, 0.0, -0.00011668027936592602, 0.0, 0.0, -3.00000090000027e-07, 0.0, 0.0, -0.0475423045930701, 0.0, 0.0, 0.0], "feature_range": [0, 1]}'''
params = json.loads(scaler_json)


A_cols = params["columns"]
B_cols = list(data.columns)

shared_cols = [c for c in A_cols if c in B_cols]
b_only_cols = [c for c in B_cols if c not in A_cols]
a_only_cols = [c for c in A_cols if c not in B_cols]

print("Shared columns:", len(shared_cols))
print("Only in B:", len(b_only_cols), b_only_cols)
print("Only in A (ignored):", len(a_only_cols), a_only_cols)

# Apply A scaling to shared columns
scale = pd.Series(params["scale"], index=A_cols)
min_ = pd.Series(params["min"], index=A_cols)

scaled_shared = data[shared_cols].mul(scale[shared_cols], axis=1).add(min_[shared_cols], axis=1)

# Apply local B scaling to columns that only exist in B
if b_only_cols:
    local_scaler = MinMaxScaler()
    scaled_b_only = pd.DataFrame(
        local_scaler.fit_transform(data[b_only_cols]),
        columns=b_only_cols,
        index=data.index
    )
else:
    scaled_b_only = pd.DataFrame(index=data.index)

# Combine back in B's original column order
data_scaled_B = pd.concat([scaled_shared, scaled_b_only], axis=1)
data_scaled_B = data_scaled_B[data.columns]


In [ ]:
params = json.loads(scaler_json)

col_stats = pd.DataFrame({
    "min": params["data_min"],
    "max": params["data_max"],
    "range": params["data_range"],
}, index=params["columns"])

col_stats.to_csv(DATA_DIR / "lab_scaling_from_json.csv")
col_stats

In [ ]:
data_scaled = data_scaled_B

In [ ]:
# MICE Imputer Scaled
imputer = IterativeImputer()
data_iterative_imputed = pd.DataFrame(imputer.fit_transform(data_scaled),
                                      columns = data_scaled.columns).set_index(data.index)
data_iterative_imputed.head()

In [ ]:
data_iterative_imputed = data_iterative_imputed.reset_index()
# data_iterative_imputed.to_csv(DATA_DIR / 'demo_labs_mice_imputed_scaled.csv', index=False)
data_iterative_imputed.to_csv(DATA_DIR / 'aou_demo_labs_mice_imputed_scaled_8_17_26.csv', index=False)

In [ ]:
data = data.reset_index()
data.to_csv(DATA_DIR / 'demo_labs_raw_w_missing_8_17_26.csv', index=False)